# JDAR SFT → inspection → DPO → inspection

This Kaggle notebook carries one model through the full experiment: reproduce the SFT run, inspect the resulting adapter, continue from that in-memory SFT adapter with DPO, then inspect the final adapter on exactly the same prompts.

The SFT hyperparameters below are copied from `sft_run_env/jdar-sft-finetuning.ipynb`. That notebook records `google/gemma-4-12B-it`; change only the model configuration cell when repeating the experiment for either of the other two models. The source notebook did not retain those two model IDs.

Before running, attach both Kaggle datasets and confirm `DPO_DATASET_PATH`. The default inspection prompts come from the training set only as a pipeline smoke test; replace them with held-out clauses before treating the comparison as an evaluation.

In [ ]:
%%capture
# Kept from the successful SFT notebook so SFT and DPO use one compatible runtime.
!pip3 install trl
!pip3 install unsloth==2026.7.5
!pip3 install accelerate
!pip3 install wandb
!pip install -U "transformers>=5.10.4"
!pip install "unsloth_zoo" --upgrade --force-reinstall --no-deps

In [ ]:
from datasets import load_dataset
import json
import os
import subprocess
from pathlib import Path

import pandas as pd
import torch
import transformers
import wandb
from accelerate import PartialState
from huggingface_hub import login
from unsloth import FastLanguageModel, get_chat_template
from trl import DPOConfig, DPOTrainer, SFTConfig, SFTTrainer

In [ ]:
print(f"transformers={transformers.__version__}")
import trl
print(f"trl={trl.__version__}")
print(f"torch={torch.__version__}")

## Credentials and experiment configuration

The source SFT notebook used the Kaggle secrets `HF_READ_DATASETS_TOKEN` and `WANDB_API_KEY`. A read token is sufficient for gated-model access and dataset downloads; use a write-capable token only if you later add Hub publishing.

In [ ]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_READ_DATASETS_TOKEN")
WANDB_API_KEY = user_secrets.get_secret("WANDB_API_KEY")

os.environ["WANDB_API_KEY"] = WANDB_API_KEY
os.environ["WANDB_SILENT"] = "true"
login(token=HF_TOKEN)
wandb.login(key=WANDB_API_KEY, relogin=True)

In [ ]:
# This is the exact model recorded in the SFT source notebook.
# For each of the other two prior runs, replace MODEL_NAME and CHAT_TEMPLATE
# with the values used for that model, then run this notebook from the top.
MODEL_NAME = "google/gemma-4-12B-it"
CHAT_TEMPLATE = "gemma-4"
MAX_SEQUENCE_LENGTH = 4096
LOAD_IN_4BIT = True

SFT_DATASET_PATH = "/kaggle/input/datasets/adrinorosario/sft-run-dataset/sft_dataset_local.json"
# Change this only if the attached Kaggle dataset has a different slug/path.
DPO_DATASET_PATH = "/kaggle/input/datasets/adrinorosario/dpo-run-dataset/dpo_dataset_local.json"

WORKING_DIR = Path("/kaggle/working")
SFT_OUTPUT_DIR = str(WORKING_DIR / "jdar_sft")
DPO_OUTPUT_DIR = str(WORKING_DIR / "jdar_dpo")
SFT_METRICS_PATH = WORKING_DIR / "jdar_sft_metrics.csv"
DPO_METRICS_PATH = WORKING_DIR / "jdar_dpo_metrics.csv"

In [ ]:
def get_available_gpu_ids() -> list[int]:
    """Detect visible CUDA devices without initializing a CUDA context."""
    try:
        result = subprocess.run(
            ["nvidia-smi", "--query-gpu=index", "--format=csv,noheader"],
            capture_output=True, text=True, check=True,
        )
        gpu_ids = [int(x.strip()) for x in result.stdout.strip().splitlines() if x.strip()]
        if not gpu_ids:
            raise ValueError("nvidia-smi returned no GPUs")
        return gpu_ids
    except Exception as error:
        print(f"nvidia-smi detection failed ({error}); falling back to torch.")
        return list(range(torch.cuda.device_count())) if torch.cuda.is_available() else []

AVAILABLE_GPUS = get_available_gpu_ids()
assert AVAILABLE_GPUS, "A CUDA GPU is required for this 12B QLoRA run."
device_string = f"cuda:{PartialState().local_process_index}"
print(f"Detected GPUs: {AVAILABLE_GPUS}; this process uses {device_string}")
print(f"Device name: {torch.cuda.get_device_name(0)}")

## SFT setup and training

This section preserves the source run’s LoRA targets and SFT hyperparameters. The extra validation makes a missing or wrongly attached dataset fail early.

In [ ]:
def load_sft_dataset():
    dataset = load_dataset("json", data_files=SFT_DATASET_PATH)
    required_columns = {"prompt", "response"}
    missing = required_columns - set(dataset["train"].column_names)
    if missing:
        raise ValueError(f"SFT data is missing columns: {sorted(missing)}")
    print(f"SFT rows: {len(dataset['train']):,}; columns: {dataset['train'].column_names}")
    return dataset

def load_model_tokenizer(model_name, maximum_sequence_length, load_in_4_bit, chat_template_name):
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_name,
        max_seq_length=maximum_sequence_length,
        load_in_4bit=load_in_4_bit,
        device_map={"": device_string},
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        lora_alpha=16,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=3407,
    )
    tokenizer = get_chat_template(tokenizer, chat_template=chat_template_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    return model, tokenizer

In [ ]:
def formatting_prompts_func(batch, tokenizer):
    texts = []
    for prompt, response in zip(batch["prompt"], batch["response"]):
        messages = [
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": response},
        ]
        texts.append(
            tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=False, reasoning_effort="high"
            )
        )
    return {"text": texts}

def map_text_column(example):
    return example["text"]

def setup_wandb_run_logging(project_name, run_name):
    return wandb.init(project=project_name, name=run_name)

def save_training_metrics(trainer, output_path):
    pd.DataFrame(trainer.state.log_history).to_csv(output_path, index=False)
    print(f"Saved metrics to {output_path}")

In [ ]:
def sft_training_setup():
    raw_dataset = load_sft_dataset()
    model, tokenizer = load_model_tokenizer(
        model_name=MODEL_NAME,
        maximum_sequence_length=MAX_SEQUENCE_LENGTH,
        load_in_4_bit=LOAD_IN_4BIT,
        chat_template_name=CHAT_TEMPLATE,
    )
    formatted_dataset = raw_dataset.map(
        lambda batch: formatting_prompts_func(batch, tokenizer), batched=True
    )
    training_args = SFTConfig(
        output_dir=SFT_OUTPUT_DIR,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        gradient_checkpointing=True,
        warmup_steps=5,
        max_steps=120,
        learning_rate=2e-4,
        fp16=False,
        bf16=False,
        logging_steps=1,
        average_tokens_across_devices=False,
        max_length=5120,
        ddp_find_unused_parameters=False,
        report_to="wandb",
        eval_strategy="no",
        eval_steps=10,
        per_device_eval_batch_size=4,
    )
    return raw_dataset, formatted_dataset, model, tokenizer, training_args

sft_raw_dataset, sft_dataset, model, tokenizer, sft_training_args = sft_training_setup()

In [ ]:
def run_sft(dataset, model, tokenizer, formatting_function, training_args, run_name):
    setup_wandb_run_logging(project_name="kaggle-sft-research", run_name=run_name)
    trainer = SFTTrainer(
        model=model,
        processing_class=tokenizer,
        train_dataset=dataset["train"],
        formatting_func=formatting_function,
        args=training_args,
    )
    trainer.train()
    wandb.finish()
    return trainer

sft_trainer = run_sft(
    dataset=sft_dataset,
    model=model,
    tokenizer=tokenizer,
    formatting_function=map_text_column,
    training_args=sft_training_args,
    run_name=MODEL_NAME.replace("/", "_") + "-sft",
)
save_training_metrics(sft_trainer, SFT_METRICS_PATH)

## Inspect the SFT adapter

Use the same prompt list again after DPO so the before/after table is directly comparable. The prefilled examples are deliberately labelled as a smoke test because they are from the training dataset.

In [ ]:
# Replace these with held-out clauses for a meaningful quality comparison.
INSPECTION_PROMPTS = sft_raw_dataset["train"].select(range(min(3, len(sft_raw_dataset["train"]))))["prompt"]

def generate_response(model, tokenizer, prompt, max_new_tokens=512):
    messages = [{"role": "user", "content": prompt}]
    input_ids = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to(next(model.parameters()).device)
    with torch.inference_mode():
        output_ids = model.generate(
            input_ids=input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(output_ids[0, input_ids.shape[-1]:], skip_special_tokens=True).strip()

def inspect_model_outputs(model, tokenizer, prompts, stage):
    FastLanguageModel.for_inference(model)
    rows = [
        {"stage": stage, "prompt": prompt, "response": generate_response(model, tokenizer, prompt)}
        for prompt in prompts
    ]
    return pd.DataFrame(rows)

In [ ]:
sft_inspection = inspect_model_outputs(model, tokenizer, INSPECTION_PROMPTS, stage="after_sft")
display(sft_inspection)

# Switch Unsloth back to training mode before constructing DPOTrainer.
FastLanguageModel.for_training(model)

## DPO setup and training

The local DPO dataset already has the standard TRL preference columns: `prompt`, `chosen`, and `rejected`. Because the SFT run used a Gemma chat template, the prompt is converted to the same user-to-assistant generation prefix before DPO; the preferred and rejected completions remain separate for `DPOTrainer`.

`ref_model=None` intentionally tells TRL to use the SFT policy at the start of DPO as the reference, while the same LoRA adapter is updated as the policy. This avoids reinitializing from the base model.

In [ ]:
def load_dpo_dataset():
    dataset = load_dataset("json", data_files=DPO_DATASET_PATH)
    required_columns = {"prompt", "chosen", "rejected"}
    missing = required_columns - set(dataset["train"].column_names)
    if missing:
        raise ValueError(f"DPO data is missing columns: {sorted(missing)}")
    empty_counts = {name: sum(not value.strip() for value in dataset["train"][name]) for name in required_columns}
    if any(empty_counts.values()):
        raise ValueError(f"DPO data contains blank values: {empty_counts}")
    print(f"DPO rows: {len(dataset['train']):,}; columns: {dataset['train'].column_names}")
    return dataset

def prepare_dpo_dataset(dataset, tokenizer):
    tokenizer.padding_side = "left"
    eos = tokenizer.eos_token or ""

    def format_preference_batch(batch):
        prompts = []
        for prompt in batch["prompt"]:
            messages = [{"role": "user", "content": prompt}]
            prompts.append(
                tokenizer.apply_chat_template(
                    messages, tokenize=False, add_generation_prompt=True, reasoning_effort="high"
                )
            )
        return {
            "prompt": prompts,
            "chosen": [text if not eos or text.endswith(eos) else text + eos for text in batch["chosen"]],
            "rejected": [text if not eos or text.endswith(eos) else text + eos for text in batch["rejected"]],
        }

    return dataset.map(format_preference_batch, batched=True)

dpo_raw_dataset = load_dpo_dataset()
dpo_dataset = prepare_dpo_dataset(dpo_raw_dataset, tokenizer)
print(dpo_dataset["train"][0])

In [ ]:
dpo_training_args = DPOConfig(
    output_dir=DPO_OUTPUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    warmup_steps=5,
    max_steps=120,
    # TRL recommends approximately 1e-5 when training adapters with DPO.
    learning_rate=1e-5,
    fp16=False,
    bf16=False,
    logging_steps=1,
    average_tokens_across_devices=False,
    max_length=5120,
    beta=0.1,
    ddp_find_unused_parameters=False,
    report_to="wandb",
    eval_strategy="no",
)

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=dpo_training_args,
    train_dataset=dpo_dataset["train"],
    processing_class=tokenizer,
)

In [ ]:
setup_wandb_run_logging(
    project_name="kaggle-dpo-research",
    run_name=MODEL_NAME.replace("/", "_") + "-dpo-from-sft",
)
dpo_trainer.train()
wandb.finish()
save_training_metrics(dpo_trainer, DPO_METRICS_PATH)

## Inspect and save the DPO adapter

The comparison table preserves both response sets for the same prompts. Review it qualitatively; it is not a substitute for a held-out, category-stratified evaluation.

In [ ]:
dpo_inspection = inspect_model_outputs(dpo_trainer.model, tokenizer, INSPECTION_PROMPTS, stage="after_dpo")
comparison = sft_inspection.rename(columns={"response": "sft_response"}).drop(columns="stage")
comparison["dpo_response"] = dpo_inspection["response"]
display(comparison)

comparison_path = WORKING_DIR / "jdar_sft_vs_dpo_responses.csv"
comparison.to_csv(comparison_path, index=False)
print(f"Saved response comparison to {comparison_path}")

In [ ]:
FINAL_ADAPTER_DIR = WORKING_DIR / "jdar_dpo_adapter"
dpo_trainer.save_model(str(FINAL_ADAPTER_DIR))
tokenizer.save_pretrained(str(FINAL_ADAPTER_DIR))
print(f"Saved final DPO adapter and tokenizer to {FINAL_ADAPTER_DIR}")